In [ ]:
!pip install rouge
!pip install transformers

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving Taylor_Swift_Genius_Data.xlsx to Taylor_Swift_Genius_Data.xlsx


In [ ]:
import os
import re
from pathlib import Path
import pandas as pd

albums = []
songs = []
lyrics = []

# Assuming ORIGINAL_ALBUMS is a list of album names you want to filter by
# For example, if you want to include all albums in the excel, you might not need this filter
# Or you would define it based on your data.
ORIGINAL_ALBUMS = ["Fearless", "Red", "1989", "Reputation", "Lover", "Folklore", "Evermore", "Midnights"] # Example, adjust as needed

# The 'uploaded' variable contains a dictionary where keys are filenames and values are file contents.
# We need to extract the filename from the uploaded dictionary to read the excel file.
if uploaded:
    # Assuming only one file was uploaded and it's the Excel file
    file_name = list(uploaded.keys())[0]
    file_content = uploaded[file_name]

    # Read the Excel file into a pandas DataFrame
    df_excel = pd.read_excel(file_content)

    # Iterate through the DataFrame to extract information, assuming specific column names
    # Adjust 'Album', 'Song', 'Lyrics' to match your Excel column names
    for index, row in df_excel.iterrows():
        album_name = row['Album'] # Assuming 'Album' is a column in your excel
        song_name = row['Song Name'] # Assuming 'Song' is a column in your excel
        raw_lyrics = str(row['Lyrics']) # Assuming 'Lyrics' is a column in your excel

        if album_name in ORIGINAL_ALBUMS:
            albums.append(album_name)
            songs.append(song_name)

            # Clean up the lyrics by replacing non-standard characters
            raw_lyrics = raw_lyrics.encode('ascii', 'replace').decode().replace('?', ' ')
            raw_lyrics = re.sub('(?!)\s+', ' ', raw_lyrics)
            # Corrected escape sequence

            # Remove lyrics header
            raw_lyrics = re.sub('.*Lyrics', '', raw_lyrics)

            # Remove end characters (number + 'Embed' or number + 'KEmbed')
            raw_lyrics = re.sub('[0-9]+KEmbed', '', raw_lyrics)
            raw_lyrics = re.sub('[0-9]+Embed', '', raw_lyrics)

            # Clean up section's tags
            section_pattern = re.compile(r'(?<!\n\n)\[([^\]]+)\]') # Corrected escape sequence
            raw_lyrics = section_pattern.sub(r'\n[\1]', section_pattern.sub(r'\n[\1]', raw_lyrics))

            # Remove firsts empty lines
            empty_lines = re.compile(r'^\s*') # Corrected escape sequence
            raw_lyrics = empty_lines.sub('', raw_lyrics)

            lyrics.append(raw_lyrics)

df = pd.DataFrame({'Album': albums, 'Song': songs, 'Lyrics': lyrics})


<>:38: SyntaxWarning: invalid escape sequence '\s'
<>:38: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_4921/3018031542.py:38: SyntaxWarning: invalid escape sequence '\s'
  raw_lyrics = re.sub('(?!)\s+', ' ', raw_lyrics)
/tmp/ipykernel_4921/3018031542.py:23: FutureWarning: Passing bytes to 'read_excel' is deprecated and will be removed in a future version. To read from a byte string, wrap it in a `BytesIO` object.
  df_excel = pd.read_excel(file_content)


In [ ]:
print(df.head(3))

      Album          Song                                             Lyrics
0  Fearless       Fifteen  You take a deep breath and you walk through th...
1  Fearless        Change  And it's a sad picture, the final blow hits yo...
2  Fearless  The Best Day  I'm five years old, it's getting cold, I've go...


In [ ]:
display(df.head(3))

,Album,Song,Lyrics
0,Fearless,Fifteen,You take a deep breath and you walk through th...
1,Fearless,Change,"And it's a sad picture, the final blow hits yo..."
2,Fearless,The Best Day,"I'm five years old, it's getting cold, I've go..."


**BERT/sentiment analysis**

In [ ]:
#from BERT. modify definetly
from transformers import BertTokenizer
tokenizer = BertTokenizer.from_pretrained('google-bert/bert-base-uncased',
                                          clean_up_tokenization_spaces=True)
print(tokenizer)
tokenized = tokenizer(["I am", "trying to find my place"], padding='longest', return_tensors="pt")
print(tokenized)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

BertTokenizer(name_or_path='google-bert/bert-base-uncased', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)
{'input_ids': tensor([[ 101, 1045, 2572,  102,    0,    0,    0],
        [ 101, 2667, 2000, 2424, 2026, 2173,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0],

In [ ]:
# modify this
import torch
from torch import nn
from transformers import BertModel
pretrained_model = BertModel.from_pretrained("google-bert/bert-base-uncased")
print(pretrained_model)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

In [ ]:
# Run but DO NOT MODIFY this code

def get_bert_embedding(text, model, tokenizer, device):
    """Get BERT [CLS] embedding for a single text."""
    model.eval()
    inputs = tokenizer(text, return_tensors='pt', padding=True, truncation=True, max_length=512)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        # Extract [CLS] token embedding (first token)
        embedding = outputs.last_hidden_state[:, 0, :].squeeze()

    return embedding.cpu()

def compute_similarity(embedding1, embedding2):
    """Compute cosine similarity between two embeddings."""
    from torch.nn.functional import cosine_similarity
    return cosine_similarity(embedding1.unsqueeze(0), embedding2.unsqueeze(0)).item()

# Setup for embedding exploration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
pretrained_model.to(device)
print(f"Using: {device}")

Using: cpu


In [ ]:
reference_text = "Welcome to New York"
print(f"Choose a song: {reference_text}")

reference_embedding = get_bert_embedding(reference_text, pretrained_model, tokenizer, device)
import heapq
#pq = []
#heapq.heappush(pq, (2, "Task B"))
#heapq.heappush(pq, (1, "Task A"))
#print(heapq.heappop(pq)) # Output: (1, 'Task A')
# This is what i'm changing
# 1. Find the most similar review in the training data to the reference (highest cosine similarity)
most_similarity = -1
most_review = ""
similar_songs = []
display(df.head(5))
#priority queue of size 13 #if size of heap is less than k add score alongside next element title else
#if greater than only add if its greater than smallest score and pop off smallest and add newest
#instead of review its the lyrics
#no need for dissimilar
top_songs = 13
print(len(df))
for i in range(len(df)):
  #if i !=0:
    #print(df['Song'])
    #verses = d
    #print(i)
    rev_embedding = get_bert_embedding(df['Lyrics'][i], pretrained_model, tokenizer, device)
    sim = compute_similarity(rev_embedding, reference_embedding) #instead of inital prompt its users prompt
    #heapq.heappush(similar_songs, (sim, df['Song'][i]))
    item = (sim, df['Song'][i])
    #print(len(similar_songs))
    if (len(similar_songs) < top_songs):
      heapq.heappush(similar_songs, item)
    else:
      if (sim > similar_songs[0][0]):
        heapq.heappop(similar_songs)
        heapq.heappush(similar_songs, item)

    if (sim > most_similarity):
      most_similarity = sim
      most_review = df['Lyrics'][i]

    #if (len(similar_songs) < 13):
     # heapq.heappush(similar_songs, (sim, df['Song'][i]))
    #else:
      #check if its greater than smallest
     # smallest = heapq.heappop(similar_songs)
      #if (sim > smallest[0]):
        #heapq.heappushpop(similar_songs, (13, song['lyrics']))
       # heapq.heappush(similar_songs, (sim, df['Song'][i]))
        #print("hello")
      #else:
       # heapq.heappush(similar_songs, smallest)
#simular_songs = sorted(similar_songs, reverse=True) #now its sorted where top is most similar
#add all songs to heap, reverse order based on score, and then take top 13
print("Most similar")
print(most_similarity)
print(most_review)
#print(similar_songs)
for song in similar_songs:
  print(song)
# 3. Print the most similar review and most dissimilar review along with their similarity scores
#
# Hint: You'll need to loop through the dataset and compute similarities of the embeddings
# Note: Start the loop from index 1 to exclude the reference review itself


Reference review: Welcome to New York


,Album,Song,Lyrics
0,Fearless,Fifteen,You take a deep breath and you walk through th...
1,Fearless,Change,"And it's a sad picture, the final blow hits yo..."
2,Fearless,The Best Day,"I'm five years old, it's getting cold, I've go..."
3,Fearless,You're Not Sorry,All this time I was wasting hoping you would c...
4,Fearless,The Way I Loved You,He is sensible and so incredible And all my si...


75
Most similar
0.7683391571044922
Find myself at your door Just like all those times before I'm not sure how I got there All roads, they lead me here I imagine you are home In your room, all alone And you open your eyes into mine And everything feels better  And right before your eyes I'm breaking, no past No reasons why Just you and me This is the last time I'm asking you this Put my name at the top of your list This is the last time I'm asking you why You break my heart in the blink of an eye (Eye, eye)  You find yourself at my door Just like all those times before You wear your best apology But I was there to watch you leave And all the times I let you in Just for you to go again Disappear when you come back Everything is better  And right before your eyes I'm aching, run fast Nowhere to hide Just you and me  This is the last time I'm asking you this Put my name at the top of your list This is the last time I'm asking you why You break my heart in the blink of an eye (Eye, eye) You